In [124]:
import os
import json
import sqlite3
from logging import exception

from groq import Groq
import requests
from numpy.ma.core import empty
from simpleeval import simple_eval
from ddgs import DDGS


In [125]:
client = Groq(api_key=os.environ['llm_key'])

In [4]:
chat_completion = client.chat.completions.create(messages=[
    {
        "role": "user",
        "content": "Today we are creating an agent with ReACT.",
    }
],
    model = "llama-3.3-70b-versatile",
)

print(chat_completion.choices[0].message.content)


ReACT (Reinforcement Learning with Abstract Contextual Temporal) is a framework for building reinforcement learning agents. To create an agent with ReACT, we'll need to break down the process into several steps:

1. **Define the environment**: Identify the problem you want the agent to solve and define the environment in which it will operate. This can include specifying the state and action spaces, rewards, and any constraints.
2. **Choose a ReACT architecture**: ReACT provides several architectures to choose from, such as ReACT-MLP (Multi-Layer Perceptron) or ReACT-LSTM (Long Short-Term Memory). Select the one that best fits your problem.
3. **Design the agent's components**: The agent will consist of several components, including:
	* **State encoder**: Maps the environment state to a compact representation.
	* **Action decoder**: Maps the agent's internal state to a specific action.
	* **Critic**: Estimates the value function, which predicts the expected return for a given state.
	*

In [93]:
def get_weather(location: str) -> str:
    weather_data = {
        "Antalya": "Sunny, 28°C",
        "Karlsruhe": "Rainy, 17°C",
        "Helsinki": "Foggy, 6°C"
    }
    return weather_data.get(location, f"No information found for the city of {location}.")


In [94]:
def get_weather_mock(location: str) -> str:
    weather_data = {"Antalya": "Sunny, 28°C", "Karlsruhe": "Rainy, 17°C"}
    return weather_data.get(location, f"No mock data for {location}.")



In [126]:
def get_weather_withAPI(location: str) -> str:
    try:
        #1.step Trying to get the geocoding
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={location}&count=1&language=en&format=json"
        geo_response = requests.get(geo_url).json()

        if "results"  not in geo_response:
            return "no information found for the city of {location}."

        lat = geo_response["results"][0]["latitude"]
        lon = geo_response["results"][0]["longitude"]

        #2.step Wİth coordinates find the weather
        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        weather_response = requests.get(weather_url).json()

        current = weather_response["current_weather"]
        temperature = current["temperature"]
        wind_speed = current["windspeed"]


        return f"Current temperature in {location} is {temperature}°C. Wind speed is {wind_speed} km/h."

    except Exception as e:
        # just in case connections is established
        return f"API Error: {str(e)}"









In [127]:
def calculate(answer: str)-> str:
    try:
        answer = answer.strip()
        result = simple_eval(answer)
        return f"Calculated answer: {result}"

    except Exception as e:
        return f"Calculation Error: {str(e)}"



In [128]:
def search_web(query: str)-> str:
    try:
        results =  DDGS().text(query,max_results= 3 )
        if len(results)==0:
             return f"Search engine error: No results found for '{query}'"


        text = ""
        for result in results:
            title = result["title"]
            body = result["body"]
            text  += f"Title: {title}\nbody: {body}\n\n---\n\n"
        return text



    except Exception as e:
        return f"Seach engine error : {str(e)}"







In [129]:
connection = sqlite3.connect("memory.db",check_same_thread=False)
cursor = connection.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS profile (
 key TEXT PRIMARY KEY,
 value TEXT)
  """)
connection.commit()

In [130]:
api_tools = [
    {

    "type": "function",
    "function": {
        "name": "search_web",
        "description": "Searches the internet for real-time information based on a query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description":"The search query string."
                }
            },
            "required": ["query"]

    }
}
    },
     {

    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Find the current weather information depending on the city name",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description":"The city name  etc. Istanbul."
                }
            },
            "required": ["location"]

    }
}
    },
     {

    "type": "function",
    "function": {
        "name": "calculate",
        "description": "used for  calculating math expressions.",
        "parameters": {
            "type": "object",
            "properties": {
                "answer": {
                    "type": "string",
                    "description":"The math expression to evaluate 25-20."
                }
            },
            "required": ["answer"]

    }
}
    },

     {

    "type": "function",
    "function": {
        "name": "save_memory",
        "description": "Saves the important information based on user query.",
        "parameters": {
            "type": "object",
            "properties": {
                "key": {
                    "type": "string",
                    "description":"The term used for memory such as name,age,location."
                },
                "value": {
                    "type": "string",
                    "description":"The term that is the value for the key such as Ali,31,Karlsruhe."
                }
            },
            "required": ["key", "value"]

    }
}
    },
    {

    "type": "function",
    "function": {
        "name": "get_user_profile",
        "description": "Gets the user data from database",
        "parameters": {
            "type": "object",
            "properties": {
                "key": {
                    "type": "string",
                    "description":"The information to look up, such as 'name'."
                },

            },
            "required": ["key"]

    }
}
    }







]

In [131]:
def save_memory(key:str,value:str) ->str:
     try:

         cursor.execute("INSERT OR REPLACE INTO profile(key,value) VALUES (?,?)",(key,value))
         connection.commit()
         return f"Successfully saved {key} and {value} in memory."
     except Exception as e:
         return f"A problem occured while saving{str(e)}"




In [132]:
def get_user_profile(key:str)->str:
    try:
        cursor.execute("SELECT value FROM profile WHERE key = ?",(key,))
        value = cursor.fetchone()

        if value:
            return f"Data found in memory: {key} = {value[0]}"
        else:
            # İŞTE BURAYI DEĞİŞTİRDİK: Modele verinin olmadığını net bir şekilde söylüyoruz.
            return f"No data found in memory for the key: '{key}'. Try a different key."

    except Exception as e:
        return f"An error occured while reading the memory: {str(e)}"

In [133]:
system_prompt = """
You are an advanced AI assistant with access to several tools.
Think logically step-by-step to answer the user's questions.
If you need information you don't have, use the appropriate tool.
Once you have all the information you need, provide a clear, helpful final answer.
"""

In [142]:
def run_reAct_agent(user_query:str,tools:dict,max_steps:int = 5 ):
    messages = [


            {"role": "system","content":system_prompt},
            {"role": "user","content":user_query}

    ]

    for step in range(max_steps):
        print(f"\n--- Adım {step + 1} ---")

        response = client.chat.completions.create(
            messages=messages,
            model="openai/gpt-oss-120b",
            temperature=0.1,
            tools = api_tools,
            tool_choice = "auto",
            parallel_tool_calls=True
        )
        response_message = response.choices[0].message


        # 1 Add model's msg to the past msgs(needs to be done)
        messages.append(response_message)

        # 2 final cevaba ulaşma kontrolü
        if response_message.tool_calls:
            print("\n[System]Model wants to use tools")

            for tool_call in response_message.tool_calls:
                action = tool_call.function.name
                action_input = tool_call.function.arguments

                print(f"The chosen tool is {action}")
                print(f"The chosen action is {action_input}")

                # convert JSON to python lib so we can use funcs
                parameters = json.loads(action_input)
                print(f"The chosen parameters is {parameters}")

                if action in tools:
                    observation = tools[action](**parameters)
                    print(f"The chosen action is {observation}")

                    messages.append({

                        "role":"tool",
                        "tool_call_id":tool_call.id,
                        "name":action,
                        "content": str(observation)

                    }
                    )
                else:
                    print(f"The chosen tool is not found  {action}")

        else:
            final_answer = response_message.content
            print(f"\nFinal Answer: {final_answer}")
            return final_answer










In [143]:

my_tools_real = {
    "get_weather": get_weather_withAPI,
    "calculate": calculate,
    "search_web": search_web,
    "save_memory": save_memory,
    "get_user_profile": get_user_profile
}
print("\nANSWER:",run_reAct_agent(" what is the weather for Karlsruhe",tools=my_tools_real))


--- Adım 1 ---

[System]Model wants to use tools
The chosen tool is get_weather
The chosen action is {"location":"Karlsruhe"}
The chosen parameters is {'location': 'Karlsruhe'}
The chosen action is Current temperature in Karlsruhe is 29.5°C. Wind speed is 12.1 km/h.

--- Adım 2 ---

Final Answer: The current weather in Karlsruhe is **29.5 °C** with a wind speed of **12.1 km/h**.

ANSWER: The current weather in Karlsruhe is **29.5 °C** with a wind speed of **12.1 km/h**.


In [136]:
print("\nANSWER:",run_reAct_agent(" what is the sqaure root of 16",tools=my_tools_real))


--- Adım 1 ---

[System]Model wants to use tools
The chosen tool is calculate
The chosen action is {"answer":"sqrt(16)"}
The chosen parameters is {'answer': 'sqrt(16)'}
The chosen action is Calculation Error: Function 'sqrt' not defined, for expression 'sqrt(16)'.

--- Adım 2 ---

[System]Model wants to use tools
The chosen tool is calculate
The chosen action is {"answer":"16^(1/2)"}
The chosen parameters is {'answer': '16^(1/2)'}
The chosen action is Calculation Error: unsupported operand type(s) for ^: 'int' and 'float'

--- Adım 3 ---

[System]Model wants to use tools
The chosen tool is calculate
The chosen action is {"answer":"4*4"}
The chosen parameters is {'answer': '4*4'}
The chosen action is Calculated answer: 16

--- Adım 4 ---

Final Answer: The square root of 16 is 4.

ANSWER: The square root of 16 is 4.


In [144]:
print("\nANSWER:",run_reAct_agent(" what is the retail price of Gta 6 for Playtation 5",tools=my_tools_real))


--- Adım 1 ---

[System]Model wants to use tools
The chosen tool is search_web
The chosen action is {"query":"GTA 6 PlayStation 5 retail price"}
The chosen parameters is {'query': 'GTA 6 PlayStation 5 retail price'}
The chosen action is Title: GTA 6 Pre-Order Guide: Where to Buy, Price... | PC Game Check
body: Grand Theft Auto VI (Pre-Order). Check price on Amazon. Frequently Asked Questions. When did GTA 6 pre-orders go live? Pre-orders opened on June 25, 2026, across the PlayStation Store, the Microsoft Store, the Rockstar Games Store and participating retailers.

---

Title: GTA 6 MAKİNASI SATIN ALDIM!- PlayStation... - YouTube
body: GTA 6 oynamak PlayStation 5 30. Yıl versiyonunu aldım, ama bi sorun niye aldım? 00:00 - Neden aldım, neyi düşünerek aldım?04:14 - Playstation 5 30. yıl ver...

---

Title: Buy PS5 Consoles | PlayStation
body: PlayStation® Credit Card offer — earn a $100 statement credit Upgrade to a PS5 now for an extra $50 off. Discover More.Shop PS5® Consoles and Bun

In [138]:
print("\nANSWER:",run_reAct_agent(" my name is Ali  and my favourite game is GTA 6",tools=my_tools_real))



--- Adım 1 ---

[System]Model wants to use tools
The chosen tool is save_memory
The chosen action is {"key":"name","value":"Ali"}
The chosen parameters is {'key': 'name', 'value': 'Ali'}
The chosen action is Successfully saved name and Ali in memory.
The chosen tool is save_memory
The chosen action is {"key":"favourite_game","value":"GTA 6"}
The chosen parameters is {'key': 'favourite_game', 'value': 'GTA 6'}
The chosen action is Successfully saved favourite_game and GTA 6 in memory.

--- Adım 2 ---

[System]Model wants to use tools
The chosen tool is get_user_profile
The chosen action is {"key":"name"}
The chosen parameters is {'key': 'name'}
The chosen action is Data found in memory: name = Ali
The chosen tool is get_user_profile
The chosen action is {"key":"favourite_game"}
The chosen parameters is {'key': 'favourite_game'}
The chosen action is Data found in memory: favourite_game = GTA 6

--- Adım 3 ---

Final Answer: Your name is Ali and your favourite game is GTA 6.

ANSWER: You

In [139]:
print("\nANSWER:",run_reAct_agent(" what is my name and my favourite game ",tools=my_tools_real))


--- Adım 1 ---

[System]Model wants to use tools
The chosen tool is get_user_profile
The chosen action is {"key":"name"}
The chosen parameters is {'key': 'name'}
The chosen action is Data found in memory: name = Ali
The chosen tool is get_user_profile
The chosen action is {"key":"favourite_game"}
The chosen parameters is {'key': 'favourite_game'}
The chosen action is Data found in memory: favourite_game = GTA 6

--- Adım 2 ---

Final Answer: Your name is Ali and your favourite game is GTA 6.

ANSWER: Your name is Ali and your favourite game is GTA 6.


In [67]:
from ddgs import DDGS
results = DDGS().text("GTA 6 PS5 price", max_results=3)
print(results)

[{'title': 'GTA 6: Release Date, Price, Platforms, and Everything ...', 'href': 'https://bitsfrombytes.com/gta-6-release-date-price-platforms/', 'body': 'Mar 23, 2026 · Grand Theft Auto VI (GTA 6) is confirmed to release on November 19, 2026, for PlayStation 5 and Xbox Series X|S, at an expected price of $70–$80.'}, {'title': 'GTA 6 Launch And Rising Console Prices: 2026 Cost Guide', 'href': 'https://www.gamermarkt.com/blog/gta-6-launch-driving-console-prices-record-highs-2026/', 'body': 'May 8, 2026 · With GTA 6 set for November 19, 2026, Sony and Microsoft have pushed console prices to unprecedented levels. Playing GTA 6 on a new PS5 now costs more than double the GTA V launch setup, and the PS5 Pro pushes total cost near $1,000.'}, {'title': 'GTA 6 Pre-Order: Opens June 25, 2026 — Prices & Editions', 'href': 'https://leonidaverse.com/en/gta-6-preorder', 'body': 'Jun 25, 2026 · GTA 6 pre-orders open June 25, 2026. Price comparator: Standard, Special, Ultimate, Collector editions and 